# 03c — Carry with **Soft** (Deformable) Payload

| | |
|---|---|
| **Author** | Narayane Ribeiro Medeiros |
| **Affiliation** | Instituto Tecnológico de Aeronáutica (ITA) |
| **Advisor** | Prof. Marcos Máximo |
| **Context** | Undergraduate's defense — 2026 |

Variant of `03a_carry_normal.ipynb`. Same MDP (goal reaching + twist tracking + hugging arms + walk-terminal reset), but the rigid box payload is replaced by a **deformable lattice** of tendon-linked spheres — the composite-style "soft box" prototyped in `deformable_hack.py`.

- Payload → `SOFT_N` × `SOFT_N` × `SOFT_N` solid lattice of freejoint spheres tied by tendon springs (axial edges + every internal face diagonal).
- Ball-ball self-collision is disabled (contype/aff pair) — only balls-vs-ground interacts.
- Deformability controlled by two knobs (`SOFTNESS`, `BOUNCINESS`) in the first cell.
- New anti-deformation rewards (`lattice_shape`, `lattice_bbox`) and a `lattice_extents` observation teach the policy to carry without squishing the box.

Config lives at `config/policy_carry_soft.py`; PPO logs go to `outputs/logs/g1_carry_soft/`.

## Notebook flags and payload knobs

Boolean switches control resume/warm-start/debug modes; four soft-payload knobs
(`SOFT_N`, `SOFT_BALL_R`, `SOFTNESS`, `BOUNCINESS`) parameterise the deformable
lattice defined in `config/policy_carry_soft.py`.

In [ ]:
REUSE_TRAINING       = False    # skip training if PPO checkpoint exists
RESUME               = False    # continue from the most recent PPO checkpoint
WARM_START_FROM_WALK = True     # load walk checkpoint as initialization (HRL-style)
DIAGNOSTIC_TRUNK_FROZEN = False # freeze transferred trunk (mlp.2, mlp.4) and replace head (mlp.6)
DEBUG_LIGHT          = True     # True = 64 envs / False = 8192 envs, few iters
RUN_VIEWER           = True     # open viser viewer at localhost:8080

# Soft-payload lattice: SOFT_N^3 balls per env (N=3 → 27 fine, N=4 → 64, N=5 → 125 viewer-only)
SOFT_N          = 3        # nodes per lattice edge
SOFT_BALL_R     = 0.012    # sphere radius (m); must fit inside half of the lattice spacing
SOFTNESS        = 1.0      # >1 = squishier (divides tendon stiffness k)
BOUNCINESS      = 1.0      # >1 = more elastic rebound (divides tendon damping d)

# TODO MASS_UNIFORMITY in [0, 1]: today the startup DR splits total payload mass equally
# across balls; a future knob will interpolate from uniform (1.0) to skewed (0.0).


In [ ]:
# ── Payload preview: discretisation + corner-drop frame sequence ───────────
# Standalone MuJoCo (no mjlab, no robot) with just the payload using the knobs
# from the previous cell — drops it corner-down from 42 cm and renders 5 key
# frames so you can see what the training will carry BEFORE registering the
# mjlab task. Replaces the old standalone deformable_hack.py script.
import os
os.environ.setdefault('MUJOCO_GL', 'egl')

import math
import matplotlib.pyplot as plt
import mujoco
import numpy as np

_BOX_HALF, _TIMESTEP, _DROP_Z, _BALL_MASS = 0.105, 0.001, 0.42, 0.005
_SPACING = 2.0 * _BOX_HALF / (SOFT_N - 1)

# tendon k/d after the knobs (same defaults as config/policy_carry_soft.py)
_AXIAL_K = 400.0 / SOFTNESS; _AXIAL_D = 10.0 / BOUNCINESS
_DIAG_K  = 100.0 / SOFTNESS; _DIAG_D  =  4.0 / BOUNCINESS

# corner-down rotation (Rodrigues around (-1,1,0)/sqrt(2))
_src = np.array([1., 1., 1.]) / math.sqrt(3.)
_dst = np.array([0., 0., -1.])
_axis = np.cross(_src, _dst); _axis /= np.linalg.norm(_axis)
_theta = math.acos(float(_src @ _dst))
_K_mat = np.array([[0., -_axis[2], _axis[1]],
                   [_axis[2], 0., -_axis[0]],
                   [-_axis[1], _axis[0], 0.]])
_R = np.eye(3) + math.sin(_theta) * _K_mat + (1. - math.cos(_theta)) * (_K_mat @ _K_mat)

_spec = mujoco.MjSpec()
_spec.option.timestep = _TIMESTEP
_spec.option.integrator = mujoco.mjtIntegrator.mjINT_IMPLICIT
_spec.option.iterations = 20
_spec.option.ls_iterations = 30
_spec.nconmax, _spec.njmax = 1500, 5000

_spec.worldbody.add_geom(
    name='floor', type=mujoco.mjtGeom.mjGEOM_PLANE, size=(2., 2., 0.1),
    rgba=(0.85, 0.85, 0.88, 1.0),
    contype=1, conaffinity=2,
    solref=(0.002, 1.0), solimp=(0.98, 0.999, 1e-4, 0.5, 2.0),
)

_nodes = [(i,j,k) for i in range(SOFT_N) for j in range(SOFT_N) for k in range(SOFT_N)]
_half  = (SOFT_N - 1) * _SPACING * 0.5
_names = {}
for (ix, iy, iz) in _nodes:
    off = np.array([ix*_SPACING-_half, iy*_SPACING-_half, iz*_SPACING-_half])
    rot = _R @ off
    nm = f'n_{ix}{iy}{iz}'; _names[(ix,iy,iz)] = nm
    b = _spec.worldbody.add_body(name=nm, pos=(rot[0], rot[1], _DROP_Z + rot[2]))
    b.add_freejoint().name = f'{nm}_free'
    b.add_geom(name=f'{nm}_g', type=mujoco.mjtGeom.mjGEOM_SPHERE,
               size=(SOFT_BALL_R, 0., 0.), mass=_BALL_MASS,
               rgba=(0.85, 0.30, 0.20, 1.0),
               condim=3, friction=(1., 0.05, 0.02),
               solref=(0.005, 1.0), solimp=(0.9, 0.99, 1e-3, 0.5, 2.0),
               contype=2, conaffinity=1)
    b.add_site(name=f'{nm}_s', pos=(0., 0., 0.), size=(1e-3, 1e-3, 1e-3))

_diag_len = _SPACING * math.sqrt(2.)
_node_set = set(_nodes)
def _spring(a, b, L, k, d):
    t = _spec.add_tendon()
    t.name = f't_{a}_{b}'
    t.stiffness[0] = k
    t.damping[0] = d
    t.springlength[:] = [L, L]
    t.wrap_site(f'{a}_s'); t.wrap_site(f'{b}_s')

for (ix, iy, iz) in _nodes:
    for (dx, dy, dz) in ((1,0,0), (0,1,0), (0,0,1)):
        nb = (ix+dx, iy+dy, iz+dz)
        if nb in _node_set:
            _spring(_names[(ix,iy,iz)], _names[nb], _SPACING, _AXIAL_K, _AXIAL_D)
for const_ax, a, b in ((0,1,2), (1,0,2), (2,0,1)):
    for c_val in range(SOFT_N):
        for u in range(SOFT_N-1):
            for v in range(SOFT_N-1):
                p1=[0,0,0]; p1[const_ax]=c_val; p1[a]=u;   p1[b]=v
                p2=[0,0,0]; p2[const_ax]=c_val; p2[a]=u+1; p2[b]=v+1
                p3=[0,0,0]; p3[const_ax]=c_val; p3[a]=u;   p3[b]=v+1
                p4=[0,0,0]; p4[const_ax]=c_val; p4[a]=u+1; p4[b]=v
                _spring(_names[tuple(p1)], _names[tuple(p2)], _diag_len, _DIAG_K, _DIAG_D)
                _spring(_names[tuple(p3)], _names[tuple(p4)], _diag_len, _DIAG_K, _DIAG_D)

_model = _spec.compile()
_data  = mujoco.MjData(_model)
mujoco.mj_forward(_model, _data)

_nballs, _ntend = SOFT_N**3, _model.ntendon
print(f'Soft payload preview        :  {SOFT_N} × {SOFT_N} × {SOFT_N}  solid lattice')
print(f'  balls per env             :  {_nballs}   (radius {SOFT_BALL_R*1000:.1f} mm, {_BALL_MASS*1000:.0f} g each)')
print(f'  tendon springs            :  {_ntend}   ({3*SOFT_N**2*(SOFT_N-1)} axial + {3*SOFT_N*(SOFT_N-1)**2*2} face-diag)')
print(f'  effective stiffness       :  axial k={_AXIAL_K:.0f} N/m, diag k={_DIAG_K:.0f} N/m   (SOFTNESS={SOFTNESS:.2f})')
print(f'  effective damping         :  axial d={_AXIAL_D:.2f}, diag d={_DIAG_D:.2f}   (BOUNCINESS={BOUNCINESS:.2f})')
print(f'  nominal cube              :  {2*_BOX_HALF*100:.0f} × {2*_BOX_HALF*100:.0f} × {2*_BOX_HALF*100:.0f} cm')

_W, _H = 720, 540
_model.vis.global_.offwidth  = max(_model.vis.global_.offwidth,  _W)
_model.vis.global_.offheight = max(_model.vis.global_.offheight, _H)
_ren = mujoco.Renderer(_model, height=_H, width=_W)
_cam = mujoco.MjvCamera()
_cam.lookat[:] = np.array([0., 0., 0.18])
_cam.distance, _cam.elevation, _cam.azimuth = 1.0, -18.0, 50.0

_frame_steps = (0, 150, 270, 340, 800)
_labels = ('initial', 'mid-fall', 'before impact', 'after impact', 'settled')
_frames = {}
for step in range(_frame_steps[-1] + 1):
    if step in _frame_steps:
        _ren.update_scene(_data, camera=_cam)
        _frames[step] = _ren.render().copy()
    mujoco.mj_step(_model, _data)

_fig, _axes = plt.subplots(1, 5, figsize=(20, 4))
for _ax, _step, _label in zip(_axes, _frame_steps, _labels):
    _ax.imshow(_frames[_step]); _ax.axis('off')
    _ax.set_title(f'{_label}  (t={_step*_TIMESTEP*1000:.0f} ms)', fontsize=11)
_fig.suptitle(
    f'Soft payload — corner drop  |  SOFT_N={SOFT_N}   SOFTNESS={SOFTNESS:.2f}   BOUNCINESS={BOUNCINESS:.2f}',
    fontsize=12,
)
_fig.tight_layout()
plt.show()


# Environment Setup

Installs pinned dependencies from `config/requirements.txt` via **uv** (5-10x faster
than pip; idempotent skip of already-satisfied packages), configures MuJoCo for
headless rendering (EGL), and detects the GPU.

In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path

if sys.version_info[:2] != (3, 10):
    raise RuntimeError(f'Python 3.10 required but got {sys.version.split()[0]}')

os.environ.setdefault('MUJOCO_GL', 'egl')
os.environ['UV_HTTP_TIMEOUT'] = '600'

REQ = Path('../requirements.txt').resolve()
if not REQ.exists():
    raise FileNotFoundError(f'requirements.txt not found at {REQ}')

if shutil.which('uv') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])

subprocess.check_call([
    sys.executable, '-m', 'uv', 'pip', 'install',
    '--index-strategy', 'unsafe-best-match',
    '--extra-index-url', 'https://download.pytorch.org/whl/cu128',
    '-r', str(REQ),
])

import torch, mjlab
if torch.cuda.is_available():
    n = torch.cuda.device_count()
    print(f'GPU count : {n}')
    for i in range(n):
        p = torch.cuda.get_device_properties(i)
        print(f'  [{i}] {p.name}  |  {p.total_memory / 1e9:.1f} GB  |  sm_{p.major}{p.minor}')
    DEVICE = 'cuda:0'
    torch.cuda.set_device(0)
else:
    DEVICE = 'cpu'
    print('WARNING: no GPU detected')

try:
    subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.free',
                    '--format=csv,noheader'], check=False)
except FileNotFoundError:
    pass


# Robot Initialization

Derives `TASK_ID` + G1 joint list + home pose + carry-specific constants (box geometry,
chest height, goal range) by importing from `config/policy_carry_soft.py`. Same pattern
as walk/pick notebooks.

In [ ]:
import sys, os
from pathlib import Path
for _p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_p / 'CLAUDE.md').exists():
        os.chdir(_p); break
sys.path.insert(0, str(Path.cwd()))

from src.policies.policy_carry_soft import (
    TASK_ID,
    G1_JOINTS, G1_HOME_ROOT_POS, G1_HOME_ROOT_ROT, G1_HOME_JOINT_POS,
    BOX_HALF, BOX_HOLD_X_OFFSET, BOX_HOLD_Z, BOX_DROP_Z,
    SOFT_TOTAL_MASS_MIN, SOFT_TOTAL_MASS_MAX,
    GOAL_REACH_RADIUS, GOAL_DIST_STAGES,
    CHEST_Z,
)

print(f'Task ID              : {TASK_ID}')
print(f'Joints               : {len(G1_JOINTS)}  (derived from mjlab G1 spec)')
print(f'Home root z          : {G1_HOME_ROOT_POS[2]:.3f} m')
print()
print(f'Box (nominal)        : cube 2h = {2*BOX_HALF:.3f} m, held {BOX_HOLD_X_OFFSET:.2f} m in front of trunk at z={BOX_HOLD_Z:.2f} m')
print(f'Box drop limit       : centroid z < {BOX_DROP_Z:.2f} m → termination (dropped)')
print()
print(f'Lattice discretisation : {SOFT_N} × {SOFT_N} × {SOFT_N} = {SOFT_N**3} balls per env (SOFT_N={SOFT_N})')
print(f'  ball radius          : {SOFT_BALL_R*1000:.1f} mm')
print(f'  SOFTNESS             : {SOFTNESS:.2f}  ({"squishier" if SOFTNESS > 1 else "stiffer" if SOFTNESS < 1 else "nominal"})')
print(f'  BOUNCINESS           : {BOUNCINESS:.2f}  ({"more elastic" if BOUNCINESS > 1 else "more plastic" if BOUNCINESS < 1 else "nominal"})')
print()
print(f'Total payload DR     : [{SOFT_TOTAL_MASS_MIN:.2f}, {SOFT_TOTAL_MASS_MAX:.2f}] kg (split uniformly over {SOFT_N**3} balls for now)')
print(f'  per-ball mass DR   : [{SOFT_TOTAL_MASS_MIN/SOFT_N**3:.4f}, {SOFT_TOTAL_MASS_MAX/SOFT_N**3:.4f}] kg')
print()
print(f'Goal reach radius    : {GOAL_REACH_RADIUS:.2f} m')
print(f'Goal dist stages     :')
for stg in GOAL_DIST_STAGES:
    print(f'    step ≥ {stg["step"]:>10,}  →  range = {stg["range"]} m')


## Task Config & Registration

Full MDP (observations, actions, rewards, events, terminations, sensors) and the PPO
runner config live in `config/policy_carry_soft.py`. Editing rewards, box/goal geometry,
DR, or PPO hyperparameters means editing that file and re-running this cell.

In [ ]:
from src.policies import policy_carry_soft

# Apply notebook knobs and register the task so TrainConfig.from_task can find it
policy_carry_soft.configure(n=SOFT_N, softness=SOFTNESS, bounciness=BOUNCINESS,
                             ball_radius_m=SOFT_BALL_R)
policy_carry_soft.register()

_env = policy_carry_soft.g1_carry_soft_env_cfg(play=False)
print(f'Task registered   : {policy_carry_soft.TASK_ID}')
print(f'Entities          : {len(_env.scene.entities)} total  ({list(_env.scene.entities.keys())[:4]} + ...)')
print(f"Sensors           : {len(_env.scene.sensors)}  ({', '.join(s.name for s in _env.scene.sensors)})")
print(f'Events            : {list(_env.events.keys())}')
print(f'  (note: reset_robot_and_box replaced by reset_robot_and_soft_box; ball_mass_split replaces box_mass)')
print(f'Rewards ({len(_env.rewards)} terms) → {list(_env.rewards.keys())}')
print(f'  (new anti-deformation: lattice_shape +1.5, lattice_bbox -1.0)')
print(f'Terminations      : {list(_env.terminations.keys())}')
print(f'Curriculum        : {list(_env.curriculum.keys())}')
print(f'Sim iter/ls       : {_env.sim.mujoco.iterations}/{_env.sim.mujoco.ls_iterations}')
print(f'Sim nconmax/njmax : {_env.sim.nconmax}/{_env.sim.njmax}')
print(f'Episode length    : {_env.episode_length_s:.2f} s')


# Simulation Parameters

Global run settings shared by training and play: task ID, number of parallel
environments, sim/decimation, episode length, and PPO hyperparameters.

In [ ]:
from pathlib import Path
import sys; _root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'CLAUDE.md').exists()); import os as _os; _os.chdir(_root); sys.path.insert(0, str(_root))
from src.utils import find_latest_checkpoint

OUT = Path('outputs'); OUT.mkdir(exist_ok=True)
MEDIA = OUT / 'media'; MEDIA.mkdir(exist_ok=True)

EXPERIMENT_HINT      = 'g1_carry_soft'
WALK_EXPERIMENT_HINT = 'g1_walk'  # used to find the walk checkpoint for warm-start
LOG_ROOT             = str(OUT / 'logs')

NUM_ENVS         = 8192
MAX_ITERATIONS   = 1000

SIM_TIMESTEP     = 0.002    # soft cfg forces 2 ms for tendon stability
DECIMATION       = 12       # keeps ~40 Hz control at 2 ms sim step
EPISODE_LENGTH_S = 8.0

# PPO / RSL-RL — carry uses a slightly lower LR than walk
LEARNING_RATE    = 3e-4
CLIP_PARAM       = 0.2
GAMMA            = 0.99
LAMBDA_GAE       = 0.95
ENTROPY_COEF     = 0.01

if DEBUG_LIGHT:
    NUM_ENVS, MAX_ITERATIONS = 64, 3

print(f'Task            : {TASK_ID}')
print(f'Parallel envs   : {NUM_ENVS}')
print(f'PPO iterations  : {MAX_ITERATIONS}')
print(f'Policy freq     : {1 / (SIM_TIMESTEP * DECIMATION):.0f} Hz')
print(f'Episode length  : {EPISODE_LENGTH_S:.2f} s')
print(f'Outputs dir     : {OUT.resolve()}')
print(f'Media dir       : {MEDIA.resolve()}')
print(f'Log root        : {Path(LOG_ROOT).resolve()}')


## Walk-Terminal Buffer

Carry resets (inherited from pick) sample each episode from a buffer of walk-terminal
states so the robot spawns in a realistic gait pose — essential for warm-start transfer,
since the actor was trained on those states. If `outputs/walk_terminal_states.npz`
exists it is reused; otherwise the trained walking policy is rolled out for a few
steps in parallel envs and snapshots are captured.

Requires a trained checkpoint in `outputs/logs/g1_velocity/`. If walking has not been
trained yet, run `01_walk.ipynb` first.

In [ ]:
import subprocess, sys
from pathlib import Path
_root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'CLAUDE.md').exists()); import os as _os; _os.chdir(_root); sys.path.insert(0, str(_root))
from src.utils import find_latest_checkpoint
from src.policies.policy_carry import WALK_BUFFER_PATH

BUFFER_PATH   = Path(WALK_BUFFER_PATH).resolve()
BUFFER_N      = 20_000
BUFFER_ENVS   = 2048 if not DEBUG_LIGHT else 128

if BUFFER_PATH.exists():
    print(f'[reuse] walk buffer already at {BUFFER_PATH}')
else:
    walk_ckpt = find_latest_checkpoint(LOG_ROOT, experiment_hint='g1_walk')
    if walk_ckpt is None:
        raise RuntimeError(
            'No walk checkpoint found in outputs/logs/g1_walk. '
            'Train the walk policy first (01_walk.ipynb).'
        )
    print(f'Walk ckpt        : {walk_ckpt}')
    print(f'Collecting {BUFFER_N} walk-terminal states across {BUFFER_ENVS} envs...')
    subprocess.check_call([
        sys.executable, '-m', 'src.buffers.warm_buffer',
        '--checkpoint', str(walk_ckpt),
        '--out',        str(BUFFER_PATH),
        '--num-envs',   str(BUFFER_ENVS),
        '--n-snapshots', str(BUFFER_N),
    ])
    print(f'Buffer written   : {BUFFER_PATH}')

## Training Setup

Build TrainConfig from the registered task and apply notebook overrides. Same pattern
as `01_walk.ipynb` (uses `TrainConfig.from_task` + `run_train` — not subprocess).

Hierarchical warm-start: when `WARM_START_FROM_WALK=True`, the setup points to the
latest `g1_velocity` checkpoint and uses the same lenient `load` monkeypatch as
`01_walk` (`strict=False`, actor+critic only), so walk weights transfer to the carry
actor's front layers even when observation dimensions change (new dims start muted).

In [ ]:
import datetime, os
from pathlib import Path
from mjlab.scripts.train import TrainConfig

cfg = TrainConfig.from_task(TASK_ID)
cfg.env.scene.num_envs    = NUM_ENVS
cfg.agent.max_iterations  = MAX_ITERATIONS
cfg.agent.logger          = 'tensorboard'
cfg.agent.experiment_name = EXPERIMENT_HINT
cfg.agent.resume          = False

log_dir = (
    Path(LOG_ROOT) / EXPERIMENT_HINT
    / datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
)

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ.setdefault('MUJOCO_GL', 'egl')

_mode = 'RESUME' if RESUME else ('WARM_START_FROM_WALK' if WARM_START_FROM_WALK else 'FROM SCRATCH')
print(f'Task            : {TASK_ID}')
print(f'Envs / iters    : {cfg.env.scene.num_envs} / {cfg.agent.max_iterations}')
print(f'Logger          : {cfg.agent.logger}')
print(f'CUDA device     : {os.environ["CUDA_VISIBLE_DEVICES"]}')
print(f'Mode            : {_mode}')
print(f'Log dir         : {log_dir.resolve()}')


## Run Training

Kicks off PPO training via `src.training.train_loop.run` (in-process, not subprocess).
Blocks the kernel until training finishes; use "Interrupt Kernel" to stop.

In [ ]:
import sys, os
from pathlib import Path
for _p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_p / 'CLAUDE.md').exists():
        os.chdir(_p); break
sys.path.insert(0, str(Path.cwd()))
from src.utils import find_latest_checkpoint
from src.training.train_loop import run as run_train_loop
from mjlab.utils.os import get_checkpoint_path

_existing_ckpt = find_latest_checkpoint(LOG_ROOT, experiment_hint=EXPERIMENT_HINT)
if REUSE_TRAINING and _existing_ckpt is not None:
    print(f'[REUSE] Checkpoint found — skipping training.\n        {_existing_ckpt}')
else:
    resume_ckpt, warm_ckpt = None, None
    if RESUME:
        resume_ckpt = get_checkpoint_path(
            log_dir.parent, r'\d{4}-\d{2}-\d{2}_.*', 'model_.*.pt',
        )
    elif WARM_START_FROM_WALK:
        warm_ckpt = find_latest_checkpoint(LOG_ROOT, experiment_hint=WALK_EXPERIMENT_HINT)
        if warm_ckpt is None:
            raise RuntimeError(
                f'WARM_START_FROM_WALK=True but no walk checkpoint found in '
                f'{LOG_ROOT}/{WALK_EXPERIMENT_HINT}. Run 01_walk.ipynb first.'
            )

    run_train_loop(
        task_id=TASK_ID,
        cfg=cfg,
        log_dir=log_dir,
        warm_start_ckpt=warm_ckpt,
        resume_ckpt=resume_ckpt,
        diagnostic_trunk_frozen=DIAGNOSTIC_TRUNK_FROZEN,
    )


## Training Curves & Robustness

Post-training analysis in two blocks: (1) training curves parsed from the TensorBoard
event file — reward, PPO diagnostics, per-term reward decomposition, terminations and
command tracking; (2) robustness — headless rollouts of the trained policy under
lateral pushes and a commanded-velocity sweep, following the RSL "Learning to Walk in
Minutes" evaluation protocol (Rudin et al., CoRL 2022).

Set `RUN_ROBUSTNESS = False` in the setup cell below to skip figures 5-6 (the
rollout-based ones).

In [ ]:
# Plot style (Wong palette + IEEEtran-like serif) and TB helpers.
import sys
from pathlib import Path
import numpy as np
import matplotlib
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

_root=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'CLAUDE.md').exists()); import os as _os; _os.chdir(_root); sys.path.insert(0, str(_root))
from src.utils import find_latest_tb_log, find_latest_checkpoint

RUN_ROBUSTNESS = True  # False skips Figs 5-6 (rollout-based)

COLORS = {
    'primary':   '#0072B2',   # Wong blue
    'accent':    '#D55E00',   # Wong vermillion
    'ok':        '#009E73',   # Wong bluish-green
    'warn':      '#E69F00',   # Wong orange
    'neutral':   '#56B4E9',
    'text':      '#1A1A1A',
    'grid':      '#B0B0B0',
}

matplotlib.rcParams.update({
    'text.usetex': False,
    'font.family': 'serif',
    'font.serif': ['Times', 'Nimbus Roman', 'Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset': 'cm',
    'font.size': 9, 'axes.labelsize': 9, 'legend.fontsize': 7.5,
    'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'axes.linewidth': 0.6, 'axes.edgecolor': COLORS['text'],
    'xtick.direction': 'in', 'ytick.direction': 'in',
    'xtick.top': True, 'ytick.right': True,
    'xtick.minor.visible': True, 'ytick.minor.visible': True,
    'grid.color': COLORS['grid'], 'grid.linewidth': 0.4, 'grid.alpha': 0.35,
    'figure.dpi': 120,
})

def load_scalar(log_dir, tag):
    """Load a TensorBoard scalar series by tag.
    Args: log_dir — path to the run directory; tag — scalar tag name.
    Returns: tuple (steps, values) as numpy arrays; empty arrays if tag missing.
    """
    ea = EventAccumulator(str(log_dir)); ea.Reload()
    if tag not in ea.Tags().get('scalars', []):
        return np.array([]), np.array([])
    evs = ea.Scalars(tag)
    return (np.array([e.step for e in evs], dtype=float),
            np.array([e.value for e in evs], dtype=float))

def list_scalars(log_dir, prefix=''):
    """List TensorBoard scalar tags filtered by a name prefix.
    Args: log_dir — path to the run directory; prefix — string prefix filter.
    Returns: list of matching tag names.
    """
    ea = EventAccumulator(str(log_dir)); ea.Reload()
    return [t for t in ea.Tags().get('scalars', []) if t.startswith(prefix)]

def smooth(x, w=15):
    """Moving-average smoothing with edge padding.
    Args: x — 1-D input array; w — window width.
    Returns: smoothed array of the same length as x.
    """
    if len(x) < w or w < 2:
        return np.asarray(x, dtype=float)
    pad = w // 2
    xp = np.pad(x, pad, mode='edge')
    return np.convolve(xp, np.ones(w) / w, mode='valid')[:len(x)]

def apply_grid(ax):
    """Apply a light two-tier grid style to a matplotlib axis.
    Args: ax — matplotlib Axes.
    Returns: none.
    """
    ax.grid(True, which='major', linewidth=0.5, alpha=0.45)
    ax.grid(True, which='minor', linewidth=0.3, alpha=0.60)
    for s in ax.spines.values():
        s.set_linewidth(0.6)
def apply_grid(ax):
    """Apply a denser two-tier grid with auto minor locators (overrides the first).
    Args: ax — matplotlib Axes.
    Returns: none.
    """
    ax.minorticks_on()
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.grid(True, which='major', linewidth=0.5, alpha=0.80)
    ax.grid(True, which='minor', linewidth=0.3, alpha=0.70)
    for s in ax.spines.values():
        s.set_linewidth(0.6)
TB_DIR = find_latest_tb_log(LOG_ROOT, experiment_hint=EXPERIMENT_HINT)
assert TB_DIR is not None, 'No TensorBoard log found — train first.'
print(f'TB dir : {TB_DIR}')


## Figure 1: training progress and PPO diagnostics

2x3 grid of TensorBoard scalars — mean episode reward, episode length, reward gain
`dR/di`, entropy, value loss (log scale), and policy std. Reads from the latest run
under `outputs/logs/g1_carry_soft/`. Saved to `outputs/media/walk_fig1_training.png`.

In [ ]:
# Fig 1: return + PPO diagnostics — 2x3 grid (reward, ep length, dR / entropy, value loss, policy std).
fig, axes = plt.subplots(2, 3, figsize=(9.6, 5.2))

# Row 1 — training progress.
x_r, y_r = load_scalar(TB_DIR, 'Train/mean_reward')
if len(y_r):
    axes[0, 0].plot(x_r, smooth(y_r), color=COLORS['primary'], lw=1.2)
axes[0, 0].set_xlabel('PPO iteration')
axes[0, 0].set_ylabel('Mean episode reward')
apply_grid(axes[0, 0])

x, y = load_scalar(TB_DIR, 'Train/mean_episode_length')
if len(y):
    axes[0, 1].plot(x, smooth(y), color=COLORS['ok'], lw=1.2)
axes[0, 1].axhline(1000, ls=':', lw=0.6, color=COLORS['text'], alpha=0.6)
axes[0, 1].text(x[-1] if len(x) else 0, 1010, 'max horizon',
                ha='right', va='bottom', fontsize=7, color=COLORS['text'], alpha=0.7)
axes[0, 1].set_xlabel('PPO iteration')
axes[0, 1].set_ylabel('Mean episode length (steps)')
apply_grid(axes[0, 1])

# Reward gain per iteration (dR/di) — how fast the policy is still improving.
if len(y_r) > 1:
    r_smooth = smooth(y_r)
    dr = np.gradient(r_smooth, x_r)
    axes[0, 2].plot(x_r, smooth(dr, w=25), color=COLORS['warn'], lw=1.2)
    axes[0, 2].axhline(0, ls=':', lw=0.6, color=COLORS['text'], alpha=0.6)
axes[0, 2].set_xlabel('PPO iteration')
axes[0, 2].set_ylabel(r'Reward gain $dR/di$')
apply_grid(axes[0, 2])

# Row 2 — PPO diagnostics.
for ax, tag, ylabel, color, log in [
    (axes[1, 0], 'Loss/entropy',    r'Entropy $H(\pi)$',        COLORS['primary'], False),
    (axes[1, 1], 'Loss/value',      r'Value loss $L_V$',        COLORS['accent'],  True),
    (axes[1, 2], 'Policy/mean_std', r'Policy std $\sigma_\pi$', COLORS['ok'],      False),
]:
    x, y = load_scalar(TB_DIR, tag)
    if len(y):
        ax.plot(x, smooth(y), color=color, lw=1.2)
    ax.set_xlabel('PPO iteration')
    ax.set_ylabel(ylabel)
    if log:
        ax.set_yscale('log')
    apply_grid(ax)

fig.tight_layout()
plt.savefig(MEDIA / 'walk_fig1_training.png', dpi=150, bbox_inches='tight')
plt.show()

if len(y_r):
    print(f'Peak reward: {np.max(smooth(y_r)):.2f}')


## Run Viewer

Loads the latest checkpoint under `outputs/logs/g1_carry_soft/` and drives the
trained policy in a single-env viser viewer. Blocks the kernel while open; use
"Interrupt Kernel" to stop.

In [ ]:
import sys, os
from pathlib import Path
for _p in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_p / 'CLAUDE.md').exists():
        os.chdir(_p); break
sys.path.insert(0, str(Path.cwd()))
from src.utils import find_latest_checkpoint
from mjlab.scripts.play import PlayConfig, run_play

PLAY_AGENT          = 'trained'   # @param ["trained","random","zero"]
PLAY_NUM_ENVS       = 1           # @param {type:"integer"}
PLAY_VIEWER         = 'viser'     # @param ["viser","native"]
CHECKPOINT_OVERRIDE = ''          # @param {type:"string"}  leave blank for latest

checkpoint = (
    Path(CHECKPOINT_OVERRIDE)
    if CHECKPOINT_OVERRIDE.strip()
    else find_latest_checkpoint(LOG_ROOT, experiment_hint=EXPERIMENT_HINT)
)

if PLAY_AGENT == 'trained' and checkpoint is None:
    raise RuntimeError('No checkpoint found. Train first or set CHECKPOINT_OVERRIDE.')

play_cfg = PlayConfig(
    agent=PLAY_AGENT,
    checkpoint_file=str(checkpoint) if (PLAY_AGENT == 'trained' and checkpoint) else None,
    num_envs=PLAY_NUM_ENVS,
    viewer=PLAY_VIEWER,
)

print(f'Agent      : {PLAY_AGENT}')
print(f'Viewer     : {PLAY_VIEWER}')
print(f'Num envs   : {PLAY_NUM_ENVS}')
if checkpoint:
    print(f'Checkpoint : {checkpoint}')


if RUN_VIEWER:
    print('Viewer starting — open http://localhost:8080 in your browser (viser).')
    print('Press Interrupt Kernel in the toolbar to close it.')
    run_play(TASK_ID, play_cfg)
else:
    print('Set RUN_VIEWER=True to open the trained policy.')